<a href="https://colab.research.google.com/github/varba187/RAGs-to-Riches/blob/main/code/notebooks/rag_token_train_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
!pip -q install transformers "datasets<3" sentencepiece accelerate faiss-cpu

In [25]:
import re
import torch
import faiss
import numpy as np
import pandas as pd

from datasets import load_dataset, Dataset
from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer,
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    BartTokenizer,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device set to {device}")

torch.manual_seed(0)

Device set to cuda


# Load NQ dataset and split for train/eval

In [26]:
nq = load_dataset("sentence-transformers/natural-questions", split="train[:5000]")
split_dataset = nq.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
dataset_name = "natural-questions"

# Helper functions

In [27]:
def get_question(example):
  return example["query"]

def get_answer(example):
  answer = example["answer"]
  if isinstance(answer, list):
    answer = answer[0]
  return answer

# Normalization function

In [28]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

# Retrieval Models

In [29]:
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = question_encoder.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-single-nq-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
context_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = context_encoder.to(device)

In [ ]:
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")
bart_model = bart_model.to(device)

# Training arguments

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./rag_token_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    logging_strategy="steps",
    logging_steps=100,
    predict_with_generate=True,
    weight_decay=0.01,
    fp16=False,
    save_total_limit=1,
    report_to="none"
)

# Question encoding

In [ ]:
def encode_question(question):
  inputs = question_tokenizer(question, return_tensors="pt", truncation=True, padding=True, max_length=128)
  inputs = {k: v.to(device) for k, v in inputs.items()}
  with torch.no_grad():
    outputs = question_encoder(**inputs).pooler_output
  return outputs.squeeze(0).cpu().numpy().astype(np.float32)

# Passage encoding

In [ ]:
def encode_passage(passage):
  inputs = context_tokenizer(passage, return_tensors="pt", truncation=True, padding=True, max_length=128)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  with torch.no_grad():
    outputs = context_encoder(**inputs).pooler_output

  return outputs.squeeze(0).cpu().numpy().astype(np.float32)

# Token-style retrieval

In [ ]:
def build_token_input(question, relevant_passages):
  chunks = []
  for i, passage in enumerate(relevant_passages):
    chunks.append(f"doc{i+1}: {passage}")
  context = " ".join(chunks)
  return f"question: {question} context: {context}"

In [ ]:
def build_passage_corpus(dataset, answer_field_fn):
  passages = []
  for i in range(len(dataset)):
      answer = answer_field_fn(dataset[i])
      if isinstance(answer, list):
        answer = answer[0]
      passages.append(answer)
  passages = list(dict.fromkeys(passages))
  return passages


# Building FAISS index

In [ ]:
def build_faiss_index(passages):
  passages_embeddings = np.stack([encode_passage(passage) for passage in passages]).astype(np.float32)
  faiss.normalize_L2(passages_embeddings)
  index = faiss.IndexFlatIP(passages_embeddings.shape[1])
  index.add(passages_embeddings)
  return index

# Building retrieval corpus for a dataset

In [ ]:
def retrieval_top_k(question, index, passages, k = 5):
  question_embedding = encode_question(question)
  question_embedding = question_embedding.reshape(1, -1)
  faiss.normalize_L2(question_embedding)
  distances, indices = index.search(question_embedding, k)
  relevant_passages = [passages[i] for i in indices[0]]
  return relevant_passages, distances[0]

In [ ]:
passages = build_passage_corpus(train_dataset, get_answer)
index = build_faiss_index(passages)

In [ ]:
def generate_token_answers(question, passages, index, k = 5):
  relevant_passages, scores = retrieval_top_k(question, index, passages, k)
  token_input = build_token_input(question, relevant_passages)

  inputs = bart_tokenizer(token_input, return_tensors="pt", truncation=True, padding=True, max_length=256)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  with torch.no_grad():
    outputs = bart_model.generate(**inputs, max_new_tokens=32, num_beams=4, no_repeat_ngram_size=2)

  answer = bart_tokenizer.decode(outputs[0], skip_special_tokens=True)
  return answer, relevant_passages

# Making examples for train/eval

In [ ]:
def make_token_examples(dataset, index, passages, k = 5):
  examples = []
  for i in range(len(dataset)):
    question = get_question(dataset[i])
    answer = get_answer(dataset[i])
    relevant_passages, scores = retrieval_top_k(question, index, passages, k)
    token_input = build_token_input(question, relevant_passages)
    examples.append({
        "input_text": token_input,
        "target_text": answer,
        "question": question
    })
  return Dataset.from_pandas(pd.DataFrame(examples))


In [ ]:
token_train = make_token_examples(train_dataset, index, passages)
token_eval = make_token_examples(eval_dataset, index, passages)

# Preprocessing for BART fine-tuning

In [ ]:
def preprocess_function(examples):
    inputs = bart_tokenizer(examples["input_text"], padding="max_length", truncation=True, max_length=256)
    outputs = bart_tokenizer(text_target=examples["target_text"], padding="max_length", truncation=True, max_length=32)
    inputs["labels"] = outputs["input_ids"]
    return inputs

tokenized_train = token_train.map(preprocess_function, remove_columns=token_train.column_names)
tokenized_eval = token_eval.map(preprocess_function, remove_columns=token_eval.column_names)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=bart_tokenizer, model=bart_model)

# EM metrics

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_pred = bart_tokenizer.batch_decode(predictions, skip_special_tokens = True)
    labels = [[token if token != -100 else bart_tokenizer.pad_token_id for token in label] for label in labels]
    decoded_labels = bart_tokenizer.batch_decode(labels, skip_special_tokens = True)

    em = [int(normalize_text(pred) == normalize_text(label)) for pred, label in zip(decoded_pred, decoded_labels)]

    return {"em": 100 * sum(em) / len(em)}

# Trainer

In [ ]:
trainer = Seq2SeqTrainer(
    model=bart_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=bart_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# Evaluate

In [ ]:
metrics = trainer.evaluate()
print(metrics)

In [ ]:
predictions = trainer.predict(tokenized_eval)

predictions, labels = predictions.predictions, predictions.label_ids
decoded_predictions = bart_tokenizer.batch_decode(predictions, skip_special_tokens=True)

labels = [[token if token != -100 else bart_tokenizer.pad_token_id for token in label] for label in labels]
decoded_labels = bart_tokenizer.batch_decode(labels, skip_special_tokens=True)

# Saving results

In [ ]:
results_df = pd.DataFrame({
    "question": token_eval["question"],
    "answer": decoded_labels,
    "prediction": decoded_predictions
})
results_df["em"]=[
    int(normalize_text(pred) == normalize_text(label))
    for pred, label in zip(results_df["prediction"], results_df["answer"])
]
results_df.to_csv("rag_token_results.csv", index=False)
print("Saved rag_token_results.csv")